# 05b_svo_edges — Directed SVO edge aggregation

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** `data/output/edges/svo_triples.jsonl` (from 05 Step 7) + `data/interim/corpus_clean.jsonl` (per-window article counts)
**Output:** `data/output/edges/svo_edges_{window}.jsonl` — one weighted **directed** actor→actor edge list per window

The SVO-track parallel of `05_edges`. Where `05_edges` aggregates **undirected** actor–concept co-occurrence, this aggregates **directed** actor→actor relations from the CAMEO-filtered SVO triples: `subject → object`, both resolved to whitelisted actors, weighted by frequency, and typed by **CAMEO quadrant** and **polarity** (conflict = negative, cooperation = positive — the same `POLARITY_COLOR` scheme as the concept edges).

Coverage is lower than co-occurrence by design — a triple only becomes an edge when it has a clean subject-verb-object parse *and* both endpoints resolve to whitelisted actors — so the directed network is sparse and **exploratory**. It feeds `06b_svo_networks`.

## Pipeline steps in this notebook

1. Setup & paths
2. Load SVO triples + per-window article counts
3. Resolve subject / object to actors (drop triples where either endpoint isn't whitelisted)
4. Aggregate directed edges (weight, CAMEO breakdown, dominant CAMEO / polarity, verbs, sources)
5. Quality report (top directed dyads, CAMEO mix per window)
6. Write svo_edges_{window}.jsonl

## Step 1: Setup & paths

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict, Counter

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM_DIR = ROOT / 'data' / 'interim'
EDGES_DIR   = ROOT / 'data' / 'output' / 'edges'

SVO_IN   = EDGES_DIR / 'svo_triples.jsonl'
CLEAN_IN = INTERIM_DIR / 'corpus_clean.jsonl'

print(f'SVO triples : {SVO_IN}')
print(f'Corpus      : {CLEAN_IN}')
assert (ROOT / 'src').is_dir(), f'ERROR: src/ not found under {ROOT}'
assert SVO_IN.exists(),   f'ERROR: {SVO_IN} not found — run 05_edges Step 7 first'
assert CLEAN_IN.exists(), f'ERROR: {CLEAN_IN} not found — run 02_clean first' 

## Step 2: Load SVO triples + per-window article counts

The article counts (from `corpus_clean.jsonl`) are the normalization denominator — the same one `05_edges` uses — so directed SVO weights are comparable across windows of different sizes.

In [ ]:
from src.alias_map    import ALIAS_MAP, ACTOR_WHITELIST
from src.verb_map     import CAMEO_POLARITY
from src.time_windows import TIME_WINDOWS

WINDOW_ORDER    = [w[0] for w in TIME_WINDOWS]
CAMEO_QUADRANTS = ['material_conflict', 'verbal_conflict',
                   'verbal_cooperation', 'material_cooperation']

with open(SVO_IN, encoding='utf-8') as f:
    triples = [json.loads(line) for line in f]
print(f'Loaded {len(triples)} SVO triples')

n_articles_per_window = Counter()
with open(CLEAN_IN, encoding='utf-8') as f:
    for line in f:
        n_articles_per_window[json.loads(line)['window']] += 1

print('Articles per window (normalization denominator):')
for w in WINDOW_ORDER:
    if w in n_articles_per_window:
        print(f'  {w:20s}  {n_articles_per_window[w]}')

## Step 3: Resolve subject / object to whitelisted actors

Same resolution rule as notebook 04 / the A4 directed graph: alias lookup → upper-snake fallback → drop `EVENT_ANCHOR` → keep only `ACTOR_WHITELIST`. A triple becomes a directed edge only when **both** subject and object resolve to (distinct) whitelisted actors — self-loops (`s == o`) are dropped.

In [ ]:
def resolve_actor(surface):
    """Surface form -> canonical whitelisted actor ID, or None."""
    canonical = ALIAS_MAP.get(surface.lower().strip())
    if canonical is None:
        canonical = surface.upper().replace(' ', '_')
    if canonical == 'EVENT_ANCHOR':
        return None
    return canonical if canonical in ACTOR_WHITELIST else None

resolved = []
n_subj_ok = n_obj_ok = n_self = 0
for t in triples:
    s = resolve_actor(t['subject'])
    o = resolve_actor(t['object'])
    n_subj_ok += s is not None
    n_obj_ok  += o is not None
    if s and o:
        if s == o:
            n_self += 1
        else:
            resolved.append((s, o, t))

pct = lambda n: f'{100 * n / max(len(triples), 1):.1f}%'
print(f'Subject resolves to a whitelisted actor : {n_subj_ok}  ({pct(n_subj_ok)})')
print(f'Object  resolves to a whitelisted actor : {n_obj_ok}  ({pct(n_obj_ok)})')
print(f'Self-loops dropped (s == o)             : {n_self}')
print(f'Actor -> actor triples (both resolve)   : {len(resolved)}  ({pct(len(resolved))})')

## Step 4: Aggregate directed edges

Key = `(window, subject, object)`. Each directed dyad accumulates a raw `weight` (pair frequency), a per-CAMEO-quadrant breakdown, the driving verbs, and the source outlets. `dominant_cameo` / `dominant_polarity` are the argmax (ties → `mixed`); `weight_normalized` divides by that window's article count.

In [ ]:
raw = defaultdict(lambda: {'weight': 0, 'cameo': Counter(),
                           'verbs': Counter(), 'sources': Counter()})
for s, o, t in resolved:
    e = raw[(t['window'], s, o)]
    e['weight'] += 1
    e['cameo'][t['cameo_category']]       += 1
    e['verbs'][t['verb_lemma']]           += 1
    e['sources'][t.get('source') or 'UNKNOWN'] += 1

def dominant(counter, tie='mixed'):
    if not counter:
        return None
    mx  = max(counter.values())
    top = [k for k, v in counter.items() if v == mx]
    return top[0] if len(top) == 1 else tie

edges_by_window = defaultdict(list)
for (window, s, o), e in raw.items():
    cam = e['cameo']
    pol = Counter()
    for quad, c in cam.items():
        pol[CAMEO_POLARITY[quad]] += c
    n_arts = n_articles_per_window.get(window, 1)
    record = {
        'subject':           s,
        'object':            o,
        'window':            window,
        'weight':            e['weight'],
        'weight_normalized': round(e['weight'] / n_arts, 4),
        'dominant_cameo':    dominant(cam),
        'dominant_polarity': dominant(pol),
        'polarity_positive': pol.get('positive', 0),
        'polarity_negative': pol.get('negative', 0),
        'verbs':             dict(e['verbs'].most_common()),
        'sources':           dict(e['sources']),
    }
    for q in CAMEO_QUADRANTS:
        record[f'cameo_{q}'] = cam.get(q, 0)
    edges_by_window[window].append(record)

n_edges = sum(len(v) for v in edges_by_window.values())
print(f'Aggregated {n_edges} directed actor->actor edges across '
      f'{len(edges_by_window)} window(s)')

## Step 5: Quality report — top directed dyads per window

In [ ]:
for window in WINDOW_ORDER:
    if window not in edges_by_window:
        continue
    ed = edges_by_window[window]
    cam_total = Counter()
    for e in ed:
        for q in CAMEO_QUADRANTS:
            cam_total[q] += e[f'cameo_{q}']
    n_arts = n_articles_per_window.get(window, 0)
    print(f'\n========== {window}  ({n_arts} articles, {len(ed)} directed dyads) ==========')
    print('  CAMEO mix: ' + '  '.join(f'{q}={cam_total[q]}' for q in CAMEO_QUADRANTS))
    print(f'  {"wt":>4}  {"polarity":>8}  subject -> object   (top verb / dominant CAMEO)')
    for e in sorted(ed, key=lambda x: -x['weight'])[:10]:
        top_verb = next(iter(e['verbs']), '')
        print(f'  {e["weight"]:4d}  {e["dominant_polarity"]:>8}  '
              f'{e["subject"]} -> {e["object"]}  ({top_verb}, {e["dominant_cameo"]})')

## Step 6: Write svo_edges_{window}.jsonl

In [ ]:
n_written = 0
for window in WINDOW_ORDER:
    if window not in edges_by_window:
        continue
    out_path = EDGES_DIR / f'svo_edges_{window}.jsonl'
    with open(out_path, 'w', encoding='utf-8') as f:
        for e in sorted(edges_by_window[window], key=lambda x: -x['weight']):
            f.write(json.dumps(e, ensure_ascii=False) + '\n')
    n_written += 1
    print(f'Wrote {len(edges_by_window[window]):3d} edges to {out_path.name}')

print(f'\nDone. {n_written} svo_edges file(s) in {EDGES_DIR}')
print()
print('VALIDATION CHECKPOINT (05b_svo_edges):')
print(f'  SVO triples in            : {len(triples)}')
print(f'  actor->actor triples      : {len(resolved)}')
print(f'  directed edges out        : {n_edges}')
print(f'  windows with edges        : {n_written}')
print()
print('NOTE: directed actor->actor SVO edges are sparse and exploratory (both')
print('      endpoints must resolve to whitelisted actors). Feeds 06b_svo_networks.')